In [42]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error
from scipy.interpolate import interp1d

In [6]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [17]:
df = pd.read_csv('../data/processed/dataset_long.csv')
df['datetime'] = pd.to_datetime(df['datetime'])
print(df.shape)
df.head(5)

(27300, 30)


,datetime,underlying_price,contract,iv,option_type,strike,hour,minute,day_of_week,date,session_progress,days_to_expiry,moneyness,log_moneyness,dist_from_atm,dist_from_atm_pct,is_ce,strike_rank,iv_neighbor_-2,iv_neighbor_-1,iv_neighbor_+1,iv_neighbor_+2,iv_neighbor_mean,wide_iv_neighbor_mean,iv_lag_1,iv_roll_mean_5,iv_roll_std_5,iv_roll_mean_10,mean_ce_iv,mean_pe_iv
0,2026-01-07 09:15:00,26111.65,NIFTY27JAN2625200CE,0.12662,CE,25200,9,15,2,2026-01-07,4860.0,20.2569,0.965086,-0.035538,911.65,0.034914,1,0.0,NaN,NaN,0.12330,0.11741,0.12330,0.11741,NaN,NaN,NaN,NaN,0.102447,0.102447
1,2026-01-07 09:20:00,26141.40,NIFTY27JAN2625200CE,0.08632,CE,25200,9,20,2,2026-01-07,4865.0,20.2535,0.963988,-0.036676,941.40,0.036012,1,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.12662,NaN,NaN,NaN,0.098723,0.098723
2,2026-01-07 09:25:00,26139.35,NIFTY27JAN2625200CE,0.09147,CE,25200,9,25,2,2026-01-07,4870.0,20.2500,0.964064,-0.036598,939.35,0.035936,1,0.0,NaN,NaN,NaN,0.09514,NaN,0.09514,0.08632,0.106470,0.028496,NaN,0.091074,0.091074
3,2026-01-07 09:30:00,26128.95,NIFTY27JAN2625200CE,0.10860,CE,25200,9,30,2,2026-01-07,4875.0,20.2465,0.964447,-0.036200,928.95,0.035553,1,0.0,NaN,NaN,0.10842,0.11150,0.10842,0.11150,0.09147,0.101470,0.021932,0.101470,0.103252,0.103252
4,2026-01-07 09:35:00,26131.90,NIFTY27JAN2625200CE,0.10462,CE,25200,9,35,2,2026-01-07,4880.0,20.2431,0.964339,-0.036313,931.90,0.035661,1,0.0,NaN,NaN,0.10538,0.12459,0.10538,0.12459,0.10860,0.103252,0.018259,0.103252,0.104097,0.104097


In [26]:
np.random.seed(42)

available_rows = df[df['iv'].notna()].index # get rows where IVs are acutally missing
hidden_rows = np.random.choice(available_rows,size=int(0.2*len(available_rows)),replace=False) # selecting rows which we're going to hide

ground_truth = df.loc[hidden_rows,'iv'] # storing actually values

df_masked = df.copy() # creating masked dataset
df_masked.loc[hidden_rows,'iv'] = np.nan

print("Total rows: ",len(df))
print("Available IV rows: ", len(available_rows))
print("Hidden rows: ", len(hidden_rows))

Total rows:  27300
Available IV rows:  21918
Hidden rows:  4383


In [36]:
# evaluation function

def eval_pred(actual, predicted):
    valid = ~np.isnan(predicted)

    mse = mean_squared_error(actual[valid],predicted[valid])

    coverage = valid.mean()

    print(f"MSE: {mse:.6f}")
    print(f"Coverage: {coverage:6f}")

    return mse, coverage

### Baseline 1: Forward fill

In [37]:
ff_df = df_masked.copy()
ff_df = ff_df.sort_values(['option_type','strike','datetime'])

ff_df['iv_ff'] = ff_df.groupby(['option_type','strike'])['iv'].ffill()

ff_pred = ff_df.loc[hidden_rows, 'iv_ff'].values

ff_mse , ff_coverage = eval_pred(ground_truth.values,ff_pred)

MSE: 0.001629
Coverage: 0.998859


### Baseline 2: Rolling mean

In [33]:
rm_pred = df_masked.loc[hidden_rows,'iv_roll_mean_5'].values

rm_mse, rm_coverage = eval_pred(ground_truth.values,rm_pred)

MSE: 0.002658
Coverage: 0.992243


### Baseline 3: Neighbor mean

In [38]:
nm_pred = df_masked.loc[hidden_rows,'iv_neighbor_mean'].values

nm_mse, nm_coverage = eval_pred(ground_truth.values,nm_pred)

MSE: 0.000266
Coverage: 0.942277


### Baseline 4: Combining forward fill + neighbor mean

In [40]:
b4_pred = np.nanmean(np.stack([ff_pred,nm_pred],axis=1),axis=1)

b4_mse, b4_coverage = eval_pred(ground_truth.values,b4_pred)

MSE: 0.000976
Coverage: 1.000000


### Baseline 5: Linear Interpolation

In [45]:
linear_pred = []

for idx in hidden_rows:
    row = df_masked.loc[idx]

    strikes_sametime = df_masked[(df_masked['datetime'] == row['datetime']) & (df_masked['option_type'] == row['option_type']) & (df_masked['iv'].notna())].sort_values('strike')

    if(len(strikes_sametime) >=3):
        interpolator = interp1d(strikes_sametime['strike'],strikes_sametime['iv'],kind='linear',bounds_error=False,fill_value='extrapolate')
        pred = float(interpolator(row['strike']))

    else: 
        pred = np.nan

    linear_pred.append(pred)

linear_pred = np.array(linear_pred)

linear_mse, linear_coverage = eval_pred(ground_truth.values,linear_pred)



MSE: 0.000098
Coverage: 0.998403


### Baseline 6: Spline Interpolation

In [49]:
spline_pred = []

for idx in hidden_rows:
    row = df_masked.loc[idx]

    strikes_sametime = df_masked[(df_masked['datetime'] == row['datetime']) & (df_masked['option_type'] == row['option_type']) & (df_masked['iv'].notna())].sort_values('strike')

    if(len(strikes_sametime)>=4):
        interpolator = interp1d(strikes_sametime['strike'],strikes_sametime['iv'],kind='cubic',bounds_error=False,fill_value='extrapolate')
        pred = float(interpolator(row['strike']))

    else:
        pred = np.nan

    spline_pred.append(pred)

spline_pred = np.array(spline_pred)

spline_mse, spline_coverage = eval_pred(ground_truth.values,spline_pred)

MSE: 0.003084
Coverage: 0.996806
